# operators_report.ipynb — отчёт операторов из CRM

Считает по каждому оператору за день: все звонки, входящие,
исходящие — и пишет в таблицу «Отчет операторов».

Файлов тут нет. Данные берутся с главного экрана CRM: ставим
фильтры и читаем число из строки «Показаны 1-20 из N записи».
Скачивать нечего, поэтому и отдельного загрузчика не нужно —
ноутбук делает всё сам.

ПРАВИЛО СЧЁТА
    Дневные операторы: все их звонки МИНУС звонки по закреплённому
    проекту. Остаётся работа на общей линии — это и есть показатель.
    Ночные операторы: всё целиком, без вычитания.

    Звонки нулевой длительности (сорвавшийся дозвон, не разговор)
    в счёт не идут — ни в общем числе, ни в том, что вычитается.
    Без этого «Все звонки» получались раздутыми в разы: за месяц
    выходило столько звонков, сколько физически принять нельзя.

Логин и пароль — в файле .env в корне репозитория:
    CRM_USER=report_user
    CRM_PASSWORD=пароль

Порядок ячеек:
    1. Настройки
    2. Вход в CRM и подсчёт
    3. Сбор данных за день
    4. Запись в Google Sheets

## ЯЧЕЙКА 1 — НАСТРОЙКИ

In [ ]:
import os
import re
import time
from datetime import date, datetime, timedelta
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv

from etl_core import SheetsLoader, log, DATA_DIR, CREDENTIALS, env, DEMO

BASE_DIR = str(DATA_DIR)   # выгрузки лежат в папке data/ в корне репозитория
# CREDENTIALS импортируется из etl_core: credentials.json в корне, в git не попадает
# --- CRM ---
IP = env("CRM_HOST", "crm.local")
BASE = f"http://{IP}/crm"

# --- таблица ---
SPREADSHEET_ID = env("OPERATORS_SHEET_ID")

# Спрашивать день и подтверждение записи при запуске.
# False — брать значения ниже молча.
ASK = True

# За какой день считать. None — вчерашний.
# Это же значение предлагается в вопросе — можно нажать Enter.
DAY = None                       # или date(2026, 8, 16)

# --- номера компаний в CRM ---
# Берутся из выпадающего списка «Компания». Похожих названий
# несколько (у проекта бывает старый и новый номер), поэтому номер
# сверяй по списку, а не по имени: перепутать их легко.
KAPPA = 111
OMEGA = 112
THETA = 108           # текущий номер проекта, а не старый
IOTA = 110

# Позиция оператора -> какие проекты вычитать.
# Пустой список = считать всё целиком, ничего не вычитая.
DAY_OPERATORS = {
    2:  [KAPPA],
    5:  [KAPPA],
    6:  [OMEGA],
    14: [OMEGA],
    25: [OMEGA],
    8:  [THETA, IOTA],
    11: [THETA, IOTA],
    12: [THETA, IOTA],
    15: [THETA, IOTA],
    16: [THETA, IOTA],
    18: [THETA, IOTA],
    20: [THETA, IOTA],
    22: [THETA, IOTA],
    26: [THETA, IOTA],
    28: [THETA, IOTA],
}

# Ночная смена: считается целиком, без вычитания проектов.
NIGHT_OPERATORS = [3, 9, 21, 24]

# Границы смен. День — полные сутки. Ночь листа за 16.08 идёт
# с 16.08 19:00 до 17.08 08:00, то есть переходит через полночь.
DAY_FROM, DAY_TO = "00:00:00", "23:59:59"
NIGHT_FROM, NIGHT_TO = "19:00:00", "08:00:00"

# Значения фильтра направления в CRM
DIR_IN, DIR_OUT = "0", "1"

# Пауза между запросами. Учётка общая с супервайзерами — не стоит
# заваливать сервер сплошной очередью.
PAUSE = 0.25

# True — только показать, что получилось, в таблицу не писать.
DRY_RUN = True

load_dotenv(DATA_DIR.parent / ".env")  # .env лежит в корне репозитория
CALLS_URL = f"{BASE}/calls/index"
LOGIN_URL = f"{BASE}/account/login"

## ЯЧЕЙКА 2 — ВХОД В CRM И ПОДСЧЁТ

In [ ]:
def get_csrf(session: requests.Session, url: str) -> str:
    """Токен из <meta name="csrf-token">. Нужен при каждом POST."""
    soup = BeautifulSoup(session.get(url, timeout=15).text, "html.parser")
    meta = soup.find("meta", {"name": "csrf-token"})
    if not meta or not meta.get("content"):
        raise RuntimeError(f"CSRF-токен не найден на {url}")
    return meta["content"]


def login() -> requests.Session:
    """Вход в CRM под профилем из .env. Возвращает готовую сессию."""
    user = os.environ.get("CRM_USER")
    password = os.environ.get("CRM_PASSWORD")
    if not user or not password:
        raise RuntimeError("Нет CRM_USER или CRM_PASSWORD — проверь .env")

    session = requests.Session()
    csrf = get_csrf(session, LOGIN_URL)
    session.post(LOGIN_URL, data={
        "_csrf": csrf,
        "LoginForm[username]": user,
        "LoginForm[password]": password,
    }, timeout=15)

    if "LoginForm" in session.get(CALLS_URL, timeout=15).text:
        raise RuntimeError("Вход не удался — проверь логин и пароль")
    print(f"Вход выполнен под {user}")
    return session


def set_period(session, start: datetime, end: datetime) -> None:
    """
    Задаёт период — это POST, а не параметр в адресе.

    После него дата держится в сессии, поэтому дальше можно слать
    сколько угодно GET-запросов с фильтрами. Для ночной смены
    период переходит через полночь, и это нормально: поля
    принимают полные даты со временем.
    """
    csrf = get_csrf(session, CALLS_URL)
    session.post(CALLS_URL, params={"Reset": 1}, data={
        "_csrf": csrf,
        "period_start": start.strftime("%Y-%m-%d %H:%M:%S"),
        "period_end": end.strftime("%Y-%m-%d %H:%M:%S"),
    }, timeout=30)


def parse_count(html: str) -> int:
    """
    Число записей из строки «Показаны 1-20 из N записи».

    Само это число и есть ответ: CRM уже посчитала, сколько строк
    подходит под фильтры. Тысячи разделены пробелом («4 971»),
    поэтому перед преобразованием выкидываем всё, кроме цифр.
    Нет блока совсем — значит ничего не нашлось, это ноль.
    """
    el = BeautifulSoup(html, "html.parser").find(class_="summary")
    if not el:
        return 0
    m = re.search(r"из\s+([\d\s\u00a0]+?)\s*запис", el.get_text(" ", strip=True))
    return int(re.sub(r"\D", "", m.group(1))) if m else 0


def count_calls(session, operator_value: str, direction: str = "",
                company: int = None, talk_time: int = None) -> int:
    """
    Один запрос — одно число.

    Все фильтры отправляем ВСЕГДА, даже пустыми. Иначе прошлый
    фильтр останется в сессии, и, скажем, «все звонки» посчитаются
    как «только по одной компании». Такую ошибку легко не заметить.
    """
    resp = session.get(CALLS_URL, params={
        "CallSearch[Operator]": operator_value,
        "CallSearch[Direction]": direction,
        "CallSearch[CompanyID]": "" if company is None else str(company),
        "CallSearch[TalkTime]": "" if talk_time is None else str(talk_time),
    }, timeout=30)
    resp.raise_for_status()
    time.sleep(PAUSE)
    return parse_count(resp.text)


def count_real(session, operator_value: str, direction: str = "",
               company: int = None) -> tuple:
    """
    То же, что count_calls, но без звонков нулевой длительности.

    В CRM 0 секунд — это не разговор, а сорвавшийся дозвон, который
    система всё равно логирует как звонок. Если их не убрать, «Все
    звонки» считает их наравне с настоящими разговорами: за месяц
    выходит счёт, который физически принять нельзя.

    Фильтра «больше нуля» в CRM нет, поэтому смотрим от противного:
    считаем все звонки, отдельно — с длительностью РОВНО 0, и
    вычитаем. TalkTime, как и остальные текстовые фильтры этой CRM
    (ID, номер), работает на точное совпадение, а не «от значения».

    Возвращает (без нулевых, сколько нулевых исключено) — второе
    число нужно, чтобы было видно масштаб поправки и можно было
    свериться с CRM руками: набрать 0 в фильтре «Длительность, сек»
    и сравнить со счётчиком.
    """
    total = count_calls(session, operator_value, direction, company)
    zero = count_calls(session, operator_value, direction, company,
                       talk_time=0)
    return total - zero, zero


def operator_values(session) -> dict:
    """
    Соответствие «номер позиции -> значение фильтра».

    В выпадающем списке подпись «Operator10», а в запрос уходит
    другое число. В таблице позиции называются «oper - 10», то есть
    номером из подписи — по нему и связываем.
    """
    soup = BeautifulSoup(session.get(CALLS_URL, timeout=15).text, "html.parser")
    sel = soup.find("select", {"name": "CallSearch[Operator]"})
    if not sel:
        raise RuntimeError("Список операторов не найден на странице")

    out = {}
    for o in sel.find_all("option"):
        m = re.fullmatch(r"\s*operator\s*(\d+)\s*", o.get_text(), re.I)
        if m and o.get("value"):
            out[int(m.group(1))] = o["value"]
    return out


# Отчёт целиком строится запросами к CRM, а в демо-режиме в сеть не ходим
if DEMO:
    raise RuntimeError("Демо-режим: отчёт операторов считается живыми "
                       "запросами к CRM — в демо он не запускается.")

session = login()
OP_VALUES = operator_values(session)
print(f"Позиций в списке CRM: {len(OP_VALUES)}")

## ЯЧЕЙКА 3 — СБОР ДАННЫХ ЗА ДЕНЬ

In [ ]:
def collect_shift(session, operators: dict, start: datetime,
                  end: datetime, title: str) -> pd.DataFrame:
    """
    Считает смену: по каждой позиции входящие, исходящие и сумму.

    Вычитание работает так: берём все звонки оператора, потом по
    каждому закреплённому проекту берём его звонки и отнимаем.
    Остаётся то, что оператор сделал вне своего проекта — работа
    на общей линии.

    Всюду считаем через count_real — без нулевых по длительности,
    иначе они попадали бы и в общий счёт, и в вычитаемый, раздувая
    оба и молча искажая итог.
    """
    set_period(session, start, end)
    print(f"\n{title}: {start:%d.%m %H:%M} — {end:%d.%m %H:%M}")

    rows = []
    for pos, subtract in operators.items():
        value = OP_VALUES.get(pos)
        if value is None:
            print(f"   Опер {pos:>2}: нет такой позиции в CRM, пропускаю")
            continue

        inc, zero_inc = count_real(session, value, DIR_IN)
        out, zero_out = count_real(session, value, DIR_OUT)
        raw_inc, raw_out = inc, out
        zero_total = zero_inc + zero_out

        for company in subtract:
            sub_inc, _ = count_real(session, value, DIR_IN, company)
            sub_out, _ = count_real(session, value, DIR_OUT, company)
            inc -= sub_inc
            out -= sub_out

        # Отрицательным показатель быть не может: если так вышло,
        # значит фильтр компании захватил больше, чем общий счёт.
        # Лучше увидеть предупреждение, чем минус в таблице.
        if inc < 0 or out < 0:
            print(f"   Опер {pos:>2}: ОТРИЦАТЕЛЬНЫЙ результат "
                  f"({raw_inc}-{raw_inc - inc}, {raw_out}-{raw_out - out}) "
                  f"— проверь список вычитаемых проектов")
            inc, out = max(inc, 0), max(out, 0)

        rows.append({"Оператор": f"oper - {pos}", "Позиция": pos,
                     "Все звонки": inc + out, "Входящий": inc,
                     "Исходящий": out})
        note = f"  (было {raw_inc}/{raw_out} до вычета проекта)" \
            if subtract else ""
        zero_note = f"  [нулевых убрано: {zero_total}]" if zero_total else ""
        print(f"   Опер {pos:>2}: всего {inc + out:>4}   вх {inc:>4}   "
              f"исх {out:>4}{note}{zero_note}")

    return pd.DataFrame(rows)


def ask_day(default: date) -> date:
    """
    Спрашивает день. Пустой ответ — значение по умолчанию.

    При кривом вводе переспрашиваем: посчитать не тот день дороже,
    чем ответить второй раз.
    """
    while True:
        raw = input(f"За какой день считать [{default:%Y-%m-%d}]: ").strip()
        if not raw:
            return default
        try:
            return datetime.strptime(raw, "%Y-%m-%d").date()
        except ValueError:
            print("   Не понял. Формат: 2026-08-16")


day = DAY or (date.today() - timedelta(days=1))
if ASK:
    day = ask_day(day)

day_df = collect_shift(
    session, DAY_OPERATORS,
    datetime.combine(day, datetime.strptime(DAY_FROM, "%H:%M:%S").time()),
    datetime.combine(day, datetime.strptime(DAY_TO, "%H:%M:%S").time()),
    "ДНЕВНАЯ СМЕНА")

night_df = collect_shift(
    session, {p: [] for p in NIGHT_OPERATORS},
    datetime.combine(day, datetime.strptime(NIGHT_FROM, "%H:%M:%S").time()),
    datetime.combine(day + timedelta(days=1),
                     datetime.strptime(NIGHT_TO, "%H:%M:%S").time()),
    "НОЧНАЯ СМЕНА")

print(f"\nИтого за {day:%d.%m.%Y}")
print(f"   день: {day_df['Все звонки'].sum()} звонков "
      f"({day_df['Входящий'].sum()} вх, {day_df['Исходящий'].sum()} исх)")
print(f"   ночь: {night_df['Все звонки'].sum()} звонков "
      f"({night_df['Входящий'].sum()} вх, {night_df['Исходящий'].sum()} исх)")

## ЯЧЕЙКА 4 — ЗАПИСЬ В GOOGLE SHEETS

In [ ]:
def write_to_sheet(day: date, day_df: pd.DataFrame,
                   night_df: pd.DataFrame, dry_run: bool = True,
                   quiet: bool = False) -> None:
    """
    Пишет числа на лист за нужную дату.

    Строки НЕ вычисляем по порядку операторов, а ищем по колонкам
    A и G: состав смены меняется, лист свёрстан руками, и слепой
    расчёт однажды запишет цифры не тому человеку. Если позиции на
    листе нет — пишем предупреждение и пропускаем, а не добавляем
    строку: дорисовывать чужую таблицу опаснее, чем недописать.
    """
    loader = SheetsLoader(CREDENTIALS, SPREADSHEET_ID)
    loader.connect()

    sheets = loader.list_date_sheets(day, day)
    if day not in sheets:
        print(f"На листе за {day:%d.%m.%Y} нет — создай его вручную")
        return
    name = sheets[day]
    q = loader.quote_sheet

    # Колонка A — дневная смена, колонка G — ночная.
    grids = loader.batch_get([f"{q(name)}!A1:A40", f"{q(name)}!G1:G40"])

    def rows_by_operator(grid):
        found = {}
        for i, r in enumerate(grid or [], start=1):
            txt = " ".join(str(r[0]).split()) if r else ""
            m = re.fullmatch(r"oper\s*-\s*(\d+)", txt, re.I)
            if m:
                found[int(m.group(1))] = i
        return found

    payload, plan = [], []
    for df, grid, first_col in ((day_df, grids[0], "B"),
                                (night_df, grids[1], "H")):
        rows = rows_by_operator(grid)
        for _, r in df.iterrows():
            row = rows.get(r["Позиция"])
            if row is None:
                if not quiet:
                    print(f"   На листе нет строки для oper - {r['Позиция']}")
                continue
            payload.append({
                "range": f"{q(name)}!{first_col}{row}",
                "values": [[int(r["Все звонки"]), int(r["Входящий"]),
                            int(r["Исходящий"])]],
            })
            plan.append((f"oper - {r['Позиция']}", f"{first_col}{row}",
                         int(r["Все звонки"])))

    if not quiet:
        print(f"\nЛист «{name}» — будет записано {len(payload)} строк:")
        for op, cell, total in plan:
            print(f"   {op:<12} {cell:<5} всего {total}")

    if dry_run:
        if not ASK:
            print("\n   Ничего не записано. Чтобы записать, "
                  "поставь DRY_RUN = False.")
        return

    loader.sh.values_batch_update({
        "valueInputOption": "USER_ENTERED",
        "data": payload,
    })
    print(f"   Записано в лист «{name}»: строк {len(payload)}.")


# Сначала показываем план, потом спрашиваем. Решение принимается
# по конкретным ячейкам, а не вслепую.
write_to_sheet(day, day_df, night_df, dry_run=True)

if ASK:
    if input("\n   Записать в таблицу? [нет/да]: ").strip().lower() \
            in ("да", "д", "yes", "y"):
        # Молча: план уже на экране, повторять его незачем.
        write_to_sheet(day, day_df, night_df, dry_run=False, quiet=True)
    else:
        print("   Отменено — в таблицу ничего не записано.")
elif not DRY_RUN:
    write_to_sheet(day, day_df, night_df, dry_run=False, quiet=True)